# Fundamental Multi-Factor Strategy Research

This notebook provides a quick view of the data, factor cross-sections and backtest results.
For the full backtest run `python scripts/run_backtest.py`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.data_loader import DataLoader
from src.utils.helpers import load_config

config = load_config()
loader = DataLoader(raw_dir=config['data']['raw_dir'], processed_dir=config['data']['processed_dir'])
print('Config loaded')

In [ ]:
# Load valuation and financial data
data = loader.load_data(config['data']['start_date'], config['data']['end_date'])
valuation, financial = data['valuation'], data['financial']
panel = loader.prepare_panel_data(valuation)
print(f"Trading days: {panel['ret'].shape[0]}, stocks: {panel['ret'].shape[1]}")
panel['close'].tail(5)

In [ ]:
# Valuation distribution of the latest cross-section
latest = pd.DataFrame({
    'pe_ttm': panel['pe_ttm'].iloc[-1],
    'pb': panel['pb'].iloc[-1],
    'total_mv': panel['total_mv'].iloc[-1] / 1e8,  # in 100M CNY
}).dropna()
latest.describe()

In [ ]:
# Backtest result summary
with open(os.path.join(config['results']['report_dir'], 'analysis_results.yaml'), encoding='utf-8') as f:
    results = yaml.safe_load(f)

pd.DataFrame(results['ic_summary']).T

In [ ]:
# NAV curves and quantile results
from IPython.display import display, Image
plot_dir = config['results']['plot_dir']
for png in ['nav_curve.png', 'quantile_nav.png', 'factor_ic_cumsum.png']:
    display(Image(filename=os.path.join(plot_dir, png)))